# Dataset

### 1. Ultilities

In [50]:
import pandas as pd
import numpy as np
import re

def extract_top_features(code):
    """
    Extract features from the input code. There are 7 features:
        1. Blank line ratio: (blank lines) / (total lines)
        2. Whitespace ratio: (spaces + 4*tabs) / (total lines)
        3. Comment ratio: (number of comment lines) / (total lines)
        4. Paragraph's lines ratio: std(lines) / (std(lines) + mean(lines))
            with 'lines' is a list of number of lines in each paragraph.
        5. Paragraph' spaces ratio: std(spaces) / (std(spaces) + mean(spaces))
            with 'spaces' is a list of number of spaces in each paragraph.
        6. Paragraph - comment ratio: (comment lines) / (comment lines + paragraphs)
        7. Standard comments: (comment with a space at the beginning) / (total comments)

    Parameters:
        code (str): The code to extract features.

    Returns:
        list of features (list[float]).
    """
    if not code or not code.strip():
        return [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

    # Feat 1: blank lines ratio
    lines = code.strip().split('\n')
    total_lines = len(lines)
    
    blank_lines = sum(1 for line in lines if not line.strip())
    blank_line_ratio = blank_lines / total_lines if total_lines > 0 else 0.0

    # Feat 2: whitespace ratio
    whitespaces = [l.count(' ') + 4 * l.count('\t') for l in lines]
    mean_whitespaces = np.mean(whitespaces) if len(whitespaces) > 0 else 0
    whitespace_ratio = np.std(whitespaces) / mean_whitespaces if mean_whitespaces > 0 else 0.0

    # Feat 3: comment ratio
    # Find '//' and '#'
    comment_lines = sum(1 for line in lines if re.search(r'(?://|#)', line))
    comment_ratio = comment_lines / total_lines if total_lines > 0 else 0.0

    # Feat 4: paragraph lines
    pr_lines = [len(pr.split('\n')) for pr in code.split("\n\n")]
    pr_lines_ratio = (np.std(pr_lines)) / (np.mean(pr_lines) + np.std(pr_lines)) if len(pr_lines) > 0 else 0.0

    # Feat 5: paragraph spaces
    paragraphs = [p for p in code.split('\n\n') if len(p) > 0]
    pr_spaces = [pr.count(' ') for pr in paragraphs]
    std_pr_spaces = np.std(pr_spaces) 
    pr_spaces_ratio = std_pr_spaces / (np.mean(pr_spaces) + std_pr_spaces) if std_pr_spaces > 0 else 0.0

    # Feat 6: paragraphs and comments
    pr_comment_ratio = comment_lines / (len(paragraphs) + comment_lines) if comment_lines > 0 else 0.0

    # Feat 7: Standard comment
    _RE_UNIVERSAL_MARKER = re.compile(r"(?:#|//|--|%)(.?)")
    comments = _RE_UNIVERSAL_MARKER.findall(code)
    if not comments:
        standard_comment_ratio = 0.0
    else:
        is_spaced = np.array([c.isspace() for c in comments])
        standard_comment_ratio = np.mean(is_spaced)

    return [
        round(blank_line_ratio, 4),
        round(whitespace_ratio, 4),
        round(comment_ratio, 4),
        round(pr_lines_ratio, 4),
        round(pr_spaces_ratio, 4),
        round(pr_comment_ratio, 4),
        round(standard_comment_ratio, 4)
    ]

def get_full_dataset(file_path: str, return_labels=True):
    """
    Read data file, extract features from data, and return features (and labels).

    Parameters:
        file_path (str): Path to parquet file
        return_labels (bool): Whether to return labels

    Returns:
        feats (np.ndarray): shape (N, num_features)
        labels (np.ndarray): shape (N,) if return_labels=True
    """
    columns = ["code", "label"] if return_labels else ["code"]
    df = pd.read_parquet(file_path, columns=columns)
    codes = df["code"].tolist()
    feats = np.array([extract_top_features(c) for c in codes])

    if (return_labels is False):
        return feats

    labels = df["label"].to_numpy()
    return feats, labels

def get_full_dataset_from_file(file_path: str):
    """
    Read features and labels from parquet file.

    Parameters:
        file_path (str): The path to parquet file.

    Returns:
        feats (np.ndarray): shape (N, num_features)
        labels (np.ndarray): shape (N,)
    """
    data = pd.read_parquet(file_path).to_numpy()
    return data[:, :-1], data[:, -1]

### 2. Extract features (if not extracted yet)

In [ ]:
# Extract features
train_feats, train_labels = get_full_dataset('../data/train.parquet')
valid_feats, valid_labels = get_full_dataset('../data/validation.parquet')
test_sample_feats, test_sample_labels = get_full_dataset('../data/test_sample.parquet')
test_feats, test_labels = get_full_dataset('../data/test.parquet', return_labels=False)

# Save extracted features to file
cols = ['blank_line','whitespace','comment', 'paragraph_lines', 'pr_spaces_ratio', 'comment_pr_ratio', 'standard_comment', 'label']

df = pd.DataFrame(np.hstack([train_feats, train_labels.reshape(-1, 1)]), columns=cols)
df.to_parquet('../extracted_features/train.parquet')

df = pd.DataFrame(np.hstack([valid_feats, valid_labels.reshape(-1, 1)]), columns=cols)
df.to_parquet('../extracted_features/validation.parquet')

df = pd.DataFrame(np.hstack([test_sample_feats, test_sample_labels.reshape(-1, 1)]), columns=cols)
df.to_parquet('../extracted_features/test_sample.parquet')

df = pd.DataFrame(np.hstack([test_feats, test_labels.reshape(-1, 1)]), columns=cols[:-1])
df.to_parquet('../extracted_features/test.parquet')

### 3. Read extracted features from files

In [51]:
train_feats, train_labels = get_full_dataset_from_file('../extracted_features/train.parquet')
valid_feats, valid_labels = get_full_dataset_from_file('../extracted_features/validation.parquet')
test_sample_feats, test_sample_labels = get_full_dataset_from_file('../extracted_features/test_sample.parquet')
test_feats = pd.read_parquet('../extracted_features/test.parquet').to_numpy()

# Model

### A. Built-in models

In [ ]:
!python -m pip install xgboost catboost lightgbm numpy

####  1. Model configuration

In [52]:
from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier, NearestCentroid

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

from sklearn.metrics import accuracy_score, f1_score

# List of models
random_forest_model = RandomForestClassifier(n_estimators=100, max_features=4)

logistic_regression_model = LogisticRegression()

knn_model = KNeighborsClassifier(n_neighbors=7, n_jobs=-1)

nc_model = NearestCentroid()

xgboost_model = XGBClassifier(
    n_estimators=1000, 
    max_depth=5, 
    learning_rate=0.15, 
    eval_metric='logloss',
)

catboost_model = CatBoostClassifier(
    iterations=1500,       
    learning_rate=0.05,
    depth=7,
    verbose=0
)

lgbm_model = lgb.LGBMClassifier(
    n_estimators=1000, 
    learning_rate=0.025,
    num_leaves=15,       
    random_state=42,
    verbosity=-1,
)

#### 2. Training

In [ ]:
_model = random_forest_model # <--- Choose a model to use
feats = [0, 1, 2, 3, 4, 5, 6] # <--- Choosing features for training,
num_train_samples = 200000 # <--- Choose number of training samples, in range (0, 500000]

_model.fit(train_feats[:num_train_samples, feats], train_labels[:num_train_samples])

CatBoostClassifier(depth=7, iterations=1500, learning_rate=0.05, verbose=0)

In [46]:
# Training score
train_pred = _model.predict(train_feats[:num_train_samples, feats])
print("Train score:", f1_score(train_pred, train_labels[:num_train_samples], average='macro'))

Train score: 0.8408086333580365


#### 4. Predict on real test and save Submission file

In [ ]:
import pandas as pd

test_predictions = _model.predict(test_feats[:, feats]).astype(int)
test_probs = _model.predict_proba(test_feats[:, feats])[:, 1]

threshold = 0.8
test_predictions = test_probs >= threshold

ids = pd.read_parquet('../data/test.parquet', columns=["ID"])["ID"].values
df = pd.DataFrame({
    "ID": ids,   
    "label": test_predictions
})

df.to_csv("submission.csv", index=False)

### B. Custom MLP

#### 1. Model configuration

In [53]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, num_feats=1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(num_feats, 16),
            nn.GELU(),
            nn.Linear(16, 8),
            nn.GELU(),
            nn.Linear(8, 4),
            nn.GELU(),
            nn.Linear(4, 2)
        )
    def forward(self, X):
        return self.fc(X)

#### 2. Train

In [54]:
def train_model(model, train_dataset: torch.tensor, valid_dataset: torch.tensor, num_feats = 3, batch_size=512, epochs=5, lr=1e-4, device="cpu"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_data_size, valid_data_size = len(train_dataset), len(valid_dataset)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        
        for i in range(0, train_data_size, batch_size):
            batch = train_dataset[i:i+batch_size]
            X, y = batch[:, :-1], batch[:, -1]
            X, y = X.to(device), y.to(device).long()

            optimizer.zero_grad()
            logits = model(X)

            # Backprop
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)

            # Check predictions
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()

        train_loss /= train_data_size
        train_acc = correct / train_data_size

        # -- Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        
        with torch.no_grad():
            for i in range(0, valid_data_size, batch_size):
                batch = valid_dataset[i:i+batch_size]
                X, y = batch[:, :-1], batch[:, -1]
                X, y = X.to(device), y.to(device).long()

                # Valid loss
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item() * X.size(0)

                # Valid accuracy
                preds = torch.argmax(logits, dim=1)
                correct += (preds == y).sum().item()
            
            val_loss /= valid_data_size
            val_acc = correct / valid_data_size

            print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

In [ ]:
feats = [0, 2, 3, 4, 6] # Choose features for training
num_train_samples = 500000
model = MLP(num_feats=len(feats)) # <-- Comment here for resuming training

_train_dataset = torch.tensor(np.hstack([train_feats[:num_train_samples, feats], train_labels[:num_train_samples].reshape(-1, 1)]), dtype=torch.float32)
_valid_dataset = torch.tensor(np.hstack([test_sample_feats[:1000, feats], test_sample_labels[:1000].reshape(-1, 1)]), dtype=torch.float32)

train_model(model, _train_dataset, _valid_dataset, device="cuda", lr=1e-4, epochs=5, batch_size=128)

Epoch 1/5 | Train Loss: 0.5748 Acc: 0.7022 | Val Loss: 0.6068 Acc: 0.7010
Epoch 2/5 | Train Loss: 0.5075 Acc: 0.7513 | Val Loss: 0.6189 Acc: 0.6980
Epoch 3/5 | Train Loss: 0.4980 Acc: 0.7540 | Val Loss: 0.6328 Acc: 0.6990
Epoch 4/5 | Train Loss: 0.4791 Acc: 0.7638 | Val Loss: 0.6666 Acc: 0.7060
Epoch 5/5 | Train Loss: 0.4486 Acc: 0.7948 | Val Loss: 0.7338 Acc: 0.6990


#### 3. Run tests

In [ ]:
def testing_predict(model, test_dataset: torch.tensor, batch_size=64, return_probs=False):
    test_data_size = len(test_dataset)

    model.eval()
    device = "cuda"

    test_predictions = []
    with torch.no_grad():
        for i in range(0, test_data_size, batch_size):
            batch = test_dataset[i:i+batch_size]
            X = batch[:, :].to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1) if return_probs is False else logits / logits.sum()
            test_predictions.extend(preds.cpu().tolist())

    return np.array(test_predictions)

##### 3a. Training score

In [ ]:
# Temp training
train_pred = testing_predict(model, torch.tensor(train_feats[:num_train_samples, feats], dtype=torch.float32))
print("Train score:", f1_score(train_pred, train_labels, average='macro'))

Train score: 0.8090771692191578


##### 3b. Real test + save submission

In [ ]:
import pandas as pd

test_feats_torch = torch.tensor(test_feats[:], dtype=torch.float32)

test_probs = testing_predict(model, test_feats_torch[:, feats], return_probs=True)

threshold = 0.8
test_predictions = test_probs[:, 1] > threshold

ids = pd.read_parquet('../data/test.parquet', columns=["ID"])["ID"].values
df = pd.DataFrame({
    "ID": ids,   
    "label": test_predictions
})

df.to_csv("submission.csv", index=False)